# Reproducibility audit before Notebook 05

Contains the 25 audit cells extracted from the reviewed 303-cell working notebook. Original cells and stored outputs are preserved in the full backup outside Git.

This is a session record, not an end-to-end reproduction. Do not Run all: download and extraction cells intentionally stop when destination files already exist.

See docs/research_reports/Pre_Notebook05_Reproducibility_Audit_v1.md for findings and the held-out evaluation policy.


In [ ]:
import sys
import platform
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Working directory:", Path.cwd())
print("Google Drive mounted:", Path("/content/drive/MyDrive").is_dir())

for package in ("torch", "torchvision", "ultralytics", "numpy", "pycocotools"):
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

Python: 3.13.16
Platform: Linux-6.6.159+-x86_64-with-glibc2.39
Working directory: /content
Google Drive mounted: False
torch: 2.14.1+cu130
torchvision: 0.29.1+cu130
ultralytics: NOT INSTALLED
numpy: 2.4.6
pycocotools: 2.0.11


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path

roots = [
    Path("/content"),
    Path("/content/drive/MyDrive"),
    Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance"),
]

keywords = ("yolo", "coco", "dataset", "fault")

for root in roots:
    print(f"\n--- {root} | exists={root.is_dir()} ---")
    if root.is_dir():
        for p in sorted(root.iterdir(), key=lambda p: p.name.lower()):
            if any(word in p.name.lower() for word in keywords):
                print("DIR " if p.is_dir() else "FILE", p.name)


--- /content | exists=True ---

--- /content/drive/MyDrive | exists=True ---
DIR  dataset
DIR  flowers_dataset
DIR  YOLOv8_Fault_Tolerance

--- /content/drive/MyDrive/YOLOv8_Fault_Tolerance | exists=True ---
DIR  datasets


In [ ]:
from pathlib import Path

paths = [
    Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance"),
    Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/datasets"),
    Path("/content/drive/MyDrive/dataset"),
]

for path in paths:
    print(f"\n--- {path} ---")
    if not path.is_dir():
        print("NOT FOUND")
        continue

    entries = sorted(path.iterdir(), key=lambda p: p.name.lower())
    for p in entries[:60]:
        print(f"{'DIR ' if p.is_dir() else 'FILE'} {p.name}")

    if len(entries) > 60:
        print(f"... {len(entries) - 60} more entries")


--- /content/drive/MyDrive/YOLOv8_Fault_Tolerance ---
DIR  artifacts
DIR  code
DIR  data
DIR  datasets
DIR  documentation
DIR  models
DIR  notebook04_git_cleanup_backup
DIR  notebook_backups

--- /content/drive/MyDrive/YOLOv8_Fault_Tolerance/datasets ---

--- /content/drive/MyDrive/dataset ---
DIR  german
DIR  italian
DIR  korean
DIR  spanish


In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance")

for name in ("data", "code"):
    path = project / name
    print(f"\n--- {path} ---")

    if not path.is_dir():
        print("NOT FOUND")
        continue

    entries = sorted(path.iterdir(), key=lambda p: p.name.lower())
    if not entries:
        print("EMPTY")

    for p in entries[:60]:
        print(f"{'DIR ' if p.is_dir() else 'FILE'} {p.name}")

    if len(entries) > 60:
        print(f"... {len(entries) - 60} more entries")


--- /content/drive/MyDrive/YOLOv8_Fault_Tolerance/data ---
DIR  single_image_test

--- /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code ---
DIR  .git
FILE .gitignore
DIR  configs
DIR  data
DIR  docs
DIR  evaluation
DIR  experiments
DIR  fault_injection
DIR  fault_tolerance
DIR  losses
DIR  models
DIR  notebooks
DIR  paper
FILE README.md
FILE requirements.txt
DIR  results
DIR  src
DIR  tests
DIR  training


In [ ]:
import subprocess
from pathlib import Path

repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")

checks = [
    ("Branch", ["git", "branch", "--show-current"]),
    ("HEAD", ["git", "log", "-1", "--format=%H%n%s%n%cI"]),
    ("Working tree", ["git", "--no-optional-locks", "status",
                      "--short", "--untracked-files=normal"]),
]

for label, command in checks:
    result = subprocess.run(
        command, cwd=repo, capture_output=True, text=True
    )
    print(f"\n--- {label} | exit={result.returncode} ---")
    print(result.stdout.strip() or "(no output)")
    if result.stderr.strip():
        print(result.stderr.strip())


--- Branch | exit=0 ---
main

--- HEAD | exit=0 ---
113d42103bcaa360ee8fe34bb98e01cde48d2119
docs: record INT4 and INT8 bit-flip unit test validation
2026-11-03T15:08:45+00:00

--- Working tree | exit=0 ---
(no output)


In [ ]:
from pathlib import Path

repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")

for name in ("data", "configs"):
    path = repo / name
    print(f"\n--- {path} ---")

    if not path.is_dir():
        print("NOT FOUND")
        continue

    entries = sorted(path.iterdir(), key=lambda p: p.name.lower())
    if not entries:
        print("EMPTY")

    for p in entries[:60]:
        print(f"{'DIR ' if p.is_dir() else 'FILE'} {p.name}")

    if len(entries) > 60:
        print(f"... {len(entries) - 60} more entries")


--- /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/data ---
EMPTY

--- /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/configs ---
EMPTY


In [ ]:
import json
import re
import subprocess
from pathlib import Path

repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
pattern = re.compile(
    r"instances_val2017|val2017|coco(?:2017)?(?:\.yaml|/|\\)|"
    r"COCO_ROOT|COCO_DIR|DATASET_ROOT",
    re.IGNORECASE,
)

tracked = subprocess.run(
    ["git", "ls-files", "-z"],
    cwd=repo, capture_output=True, text=True, check=True
).stdout.split("\0")

matches = []
for name in filter(None, tracked):
    path = repo / name
    if path.suffix.lower() not in {".py", ".ipynb", ".md", ".yaml", ".yml", ".txt"}:
        continue

    if path.suffix.lower() == ".ipynb":
        notebook = json.loads(path.read_text(encoding="utf-8"))
        sections = [
            (f"cell {i + 1}", "".join(cell.get("source", [])))
            for i, cell in enumerate(notebook["cells"])
        ]
    else:
        sections = [("file", path.read_text(encoding="utf-8"))]

    for section, text in sections:
        for line_no, line in enumerate(text.splitlines(), 1):
            if pattern.search(line):
                matches.append(
                    f"{name} | {section} | line {line_no}: {line.strip()[:400]}"
                )

print(f"Total matching lines: {len(matches)}")
for match in matches[:80]:
    print(match)
if len(matches) > 80:
    print(f"... {len(matches) - 80} additional matches not displayed")

Total matching lines: 29
notebooks/03_baseline_validation.ipynb | cell 6 | line 4: dataset_root = Path(PROJECT_ROOT) / "datasets"
notebooks/03_baseline_validation.ipynb | cell 6 | line 10: print("Dataset directory exists:", dataset_root.exists())
notebooks/03_baseline_validation.ipynb | cell 6 | line 12: if dataset_root.exists():
notebooks/03_baseline_validation.ipynb | cell 6 | line 13: for item in sorted(dataset_root.iterdir()):
notebooks/03_baseline_validation.ipynb | cell 9 | line 5: COCO_ROOT = Path("/content/datasets/coco2017")
notebooks/03_baseline_validation.ipynb | cell 9 | line 6: COCO_ROOT.mkdir(parents=True, exist_ok=True)
notebooks/03_baseline_validation.ipynb | cell 9 | line 8: zip_path = COCO_ROOT / "val2017.zip"
notebooks/03_baseline_validation.ipynb | cell 9 | line 9: images_dir = COCO_ROOT / "images"
notebooks/03_baseline_validation.ipynb | cell 9 | line 12: url = "http://images.cocodataset.org/zips/val2017.zip"
notebooks/03_baseline_validation.ipynb | cell 9 | line 3

In [ ]:
from pathlib import Path

root = Path("/content/datasets/coco2017")
images = root / "images" / "val2017"
annotations = root / "annotations" / "instances_val2017.json"
labels = root / "labels" / "val2017"
config = root / "coco2017_val.yaml"

print("COCO root exists:", root.is_dir())
print("Images directory exists:", images.is_dir())
print("JPG count:", sum(1 for _ in images.glob("*.jpg")))
print("Annotations JSON exists:", annotations.is_file())
print("Labels directory exists:", labels.is_dir())
print("TXT label count:", sum(1 for _ in labels.glob("*.txt")))
print("Dataset YAML exists:", config.is_file())

COCO root exists: False
Images directory exists: False
JPG count: 0
Annotations JSON exists: False
Labels directory exists: False
TXT label count: 0
Dataset YAML exists: False


In [ ]:
from pathlib import Path
import shutil

target = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/datasets/coco2017"
)

usage = shutil.disk_usage("/content/drive/MyDrive")
print("Drive mount reported free space (GiB):", round(usage.free / 2**30, 2))
print("Target exists:", target.exists())

if target.is_dir():
    for p in sorted(target.iterdir(), key=lambda p: p.name):
        print("DIR " if p.is_dir() else "FILE", p.name)

Drive mount reported free space (GiB): 10.55
Target exists: False


In [ ]:
from urllib.request import Request, urlopen

url = "https://s3.amazonaws.com/images.cocodataset.org/zips/val2017.zip"
request = Request(url, method="HEAD")

with urlopen(request, timeout=60) as response:
    print("HTTP status:", response.status)
    print("Final URL:", response.url)
    print("Content-Length:", response.headers.get("Content-Length"))
    print("Content-Type:", response.headers.get("Content-Type"))

HTTP status: 200
Final URL: https://s3.amazonaws.com/images.cocodataset.org/zips/val2017.zip
Content-Length: 815585330
Content-Type: application/zip


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import shutil

root = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/datasets/coco2017"
)
root.mkdir(parents=True, exist_ok=True)

destination = root / "val2017.zip"
partial = root / "val2017.zip.part"
url = "https://s3.amazonaws.com/images.cocodataset.org/zips/val2017.zip"
expected_size = 815585330

if destination.exists() or partial.exists():
    raise FileExistsError(
        "An archive or partial download already exists. Inspect it before continuing."
    )

with urlopen(url, timeout=120) as response:
    if response.status != 200:
        raise RuntimeError(f"Unexpected HTTP status: {response.status}")

    with partial.open("xb") as output:
        shutil.copyfileobj(response, output, length=8 * 1024 * 1024)

actual_size = partial.stat().st_size
print("Downloaded bytes:", actual_size)
print("Expected bytes:", expected_size)

if actual_size != expected_size:
    raise RuntimeError("Size mismatch. Partial download preserved for inspection.")

partial.rename(destination)
print("Saved:", destination)
print("ZIP integrity has not been verified yet.")

Downloaded bytes: 815585330
Expected bytes: 815585330
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/datasets/coco2017/val2017.zip
ZIP integrity has not been verified yet.


In [ ]:
from pathlib import Path
from zipfile import ZipFile

archive = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/datasets/coco2017/val2017.zip"
)

with ZipFile(archive, "r") as z:
    files = [item for item in z.infolist() if not item.is_dir()]
    images = [
        item for item in files
        if item.filename.startswith("val2017/")
        and item.filename.lower().endswith(".jpg")
    ]

    names = [item.filename for item in files]
    print("File count:", len(files))
    print("JPG count under val2017/:", len(images))
    print("Duplicate file names:", len(names) - len(set(names)))
    print(
        "Total uncompressed size (GiB):",
        round(sum(item.file_size for item in files) / 2**30, 3),
    )

    bad_file = z.testzip()
    print("CRC check:", "PASS" if bad_file is None else f"FAIL: {bad_file}")

    if bad_file is not None:
        raise RuntimeError("ZIP integrity check failed.")
    if len(images) != 5000 or len(files) != 5000:
        raise RuntimeError("Unexpected archive contents.")
    if len(names) != len(set(names)):
        raise RuntimeError("Duplicate file names detected.")

print("Archive structure and CRC checks passed.")

File count: 5000
JPG count under val2017/: 5000
Duplicate file names: 0
Total uncompressed size (GiB): 0.759
CRC check: PASS
Archive structure and CRC checks passed.


In [ ]:
from pathlib import Path
from zipfile import ZipFile

root = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/datasets/coco2017"
)
archive = root / "val2017.zip"
destination = root / "images"
image_dir = destination / "val2017"

if image_dir.exists():
    raise FileExistsError(
        "The image directory already exists. Inspect it before extracting."
    )

destination.mkdir(parents=True, exist_ok=True)

with ZipFile(archive, "r") as z:
    base = destination.resolve()
    for member in z.infolist():
        target = (base / member.filename).resolve()
        if not target.is_relative_to(base):
            raise RuntimeError(f"Unsafe archive path: {member.filename}")

    z.extractall(destination)

count = sum(1 for p in image_dir.glob("*.jpg") if p.is_file())
print("Image directory:", image_dir)
print("Extracted JPG count:", count)

if count != 5000:
    raise RuntimeError("Unexpected extracted image count.")

print("Extraction completed. Source ZIP preserved.")

Image directory: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/datasets/coco2017/images/val2017
Extracted JPG count: 5000
Extraction completed. Source ZIP preserved.


In [ ]:
from urllib.request import Request, urlopen

url = (
    "https://s3.amazonaws.com/images.cocodataset.org/"
    "annotations/annotations_trainval2017.zip"
)

request = Request(url, method="HEAD")

with urlopen(request, timeout=60) as response:
    print("HTTP status:", response.status)
    print("Final URL:", response.url)
    print("Content-Length:", response.headers.get("Content-Length"))
    print("Content-Type:", response.headers.get("Content-Type"))

HTTP status: 200
Final URL: https://s3.amazonaws.com/images.cocodataset.org/annotations/annotations_trainval2017.zip
Content-Length: 252907541
Content-Type: application/zip


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import shutil

root = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/datasets/coco2017"
)
destination = root / "annotations_trainval2017.zip"
partial = root / "annotations_trainval2017.zip.part"
expected_size = 252907541
url = (
    "https://s3.amazonaws.com/images.cocodataset.org/"
    "annotations/annotations_trainval2017.zip"
)

if destination.exists() or partial.exists():
    raise FileExistsError(
        "An archive or partial download already exists. Inspect it before continuing."
    )

with urlopen(url, timeout=120) as response:
    if response.status != 200:
        raise RuntimeError(f"Unexpected HTTP status: {response.status}")

    with partial.open("xb") as output:
        shutil.copyfileobj(response, output, length=8 * 1024 * 1024)

actual_size = partial.stat().st_size
print("Downloaded bytes:", actual_size)
print("Expected bytes:", expected_size)

if actual_size != expected_size:
    raise RuntimeError("Size mismatch. Partial download preserved for inspection.")

partial.rename(destination)
print("Saved:", destination)
print("ZIP integrity has not been verified yet.")

Downloaded bytes: 252907541
Expected bytes: 252907541
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/datasets/coco2017/annotations_trainval2017.zip
ZIP integrity has not been verified yet.


In [ ]:
from pathlib import Path
from zipfile import ZipFile

archive = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/"
    "datasets/coco2017/annotations_trainval2017.zip"
)
required = "annotations/instances_val2017.json"

with ZipFile(archive, "r") as z:
    names = z.namelist()
    print("Required annotation present:", required in names)
    print("Required annotation occurrences:", names.count(required))

    if names.count(required) != 1:
        raise RuntimeError("Expected exactly one validation instances JSON.")

    info = z.getinfo(required)
    print("Validation JSON size (bytes):", info.file_size)

    bad_file = z.testzip()
    print("CRC check:", "PASS" if bad_file is None else f"FAIL: {bad_file}")

    if bad_file is not None:
        raise RuntimeError("ZIP integrity check failed.")

print("Archive CRC passed. Required annotation is present.")

Required annotation present: True
Required annotation occurrences: 1
Validation JSON size (bytes): 19987840
CRC check: PASS
Archive CRC passed. Required annotation is present.


In [ ]:
from pathlib import Path
from zipfile import ZipFile

root = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/datasets/coco2017"
)
archive = root / "annotations_trainval2017.zip"
member = "annotations/instances_val2017.json"
destination = root / member

if destination.exists():
    raise FileExistsError(
        "The annotation file already exists. Inspect it before extracting."
    )

with ZipFile(archive, "r") as z:
    info = z.getinfo(member)
    z.extract(member, path=root)

actual_size = destination.stat().st_size
print("Annotation path:", destination)
print("Extracted bytes:", actual_size)
print("Expected bytes:", info.file_size)

if actual_size != info.file_size:
    raise RuntimeError("Extracted annotation size mismatch.")

print("Extraction completed. JSON contents have not been validated yet.")

Annotation path: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/datasets/coco2017/annotations/instances_val2017.json
Extracted bytes: 19987840
Expected bytes: 19987840
Extraction completed. JSON contents have not been validated yet.


In [ ]:
import json
from pathlib import Path

root = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/datasets/coco2017"
)

with (root / "annotations/instances_val2017.json").open(
    encoding="utf-8"
) as f:
    coco = json.load(f)

images = coco["images"]
annotations = coco["annotations"]
categories = coco["categories"]

image_ids = [item["id"] for item in images]
category_ids = [item["id"] for item in categories]
annotation_ids = [item["id"] for item in annotations]
image_id_set = set(image_ids)
category_id_set = set(category_ids)

expected_names = {item["file_name"] for item in images}
actual_names = {
    p.name for p in (root / "images/val2017").glob("*.jpg")
    if p.is_file()
}

missing = expected_names - actual_names
extra = actual_names - expected_names
invalid_image_refs = sum(
    item["image_id"] not in image_id_set for item in annotations
)
invalid_category_refs = sum(
    item["category_id"] not in category_id_set for item in annotations
)

checks = {
    "Expected image count": len(images) == 5000,
    "Expected category count": len(categories) == 80,
    "Unique image IDs": len(image_ids) == len(image_id_set),
    "Unique image filenames": len(expected_names) == len(images),
    "Unique category IDs": len(category_ids) == len(category_id_set),
    "Unique annotation IDs": len(annotation_ids) == len(set(annotation_ids)),
    "Image filenames match disk": not missing and not extra,
    "Valid annotation image references": invalid_image_refs == 0,
    "Valid annotation category references": invalid_category_refs == 0,
}

print("Images:", len(images))
print("Annotations:", len(annotations))
print("Categories:", len(categories))
print("Missing images:", len(missing))
print("Extra images:", len(extra))

for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

if not all(checks.values()):
    raise RuntimeError("Dataset consistency checks failed.")

print("JSON and filename consistency checks passed.")
print("Image decoding and dimensions have not been checked yet.")

Images: 5000
Annotations: 36781
Categories: 80
Missing images: 0
Extra images: 0
Expected image count: PASS
Expected category count: PASS
Unique image IDs: PASS
Unique image filenames: PASS
Unique category IDs: PASS
Unique annotation IDs: PASS
Image filenames match disk: PASS
Valid annotation image references: PASS
Valid annotation category references: PASS
JSON and filename consistency checks passed.
Image decoding and dimensions have not been checked yet.


In [ ]:
import json
from pathlib import Path
from PIL import Image

root = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/datasets/coco2017"
)

with (root / "annotations/instances_val2017.json").open(
    encoding="utf-8"
) as f:
    images = json.load(f)["images"]

failures = []

for index, item in enumerate(images, start=1):
    path = root / "images/val2017" / item["file_name"]
    expected_size = (item["width"], item["height"])

    try:
        with Image.open(path) as image:
            image.load()
            if image.size != expected_size:
                failures.append(
                    (item["file_name"], "dimension mismatch",
                     image.size, expected_size)
                )
            if image.format != "JPEG":
                failures.append(
                    (item["file_name"], "unexpected format", image.format)
                )
    except Exception as error:
        failures.append(
            (item["file_name"], type(error).__name__, str(error))
        )

    if index % 500 == 0:
        print(f"Checked {index}/{len(images)} images", flush=True)

print("Failure count:", len(failures))
for failure in failures[:20]:
    print(failure)

if failures:
    raise RuntimeError("Image decoding or metadata checks failed.")

print("All images decoded successfully and dimensions match the JSON.")

Checked 500/5000 images
Checked 1000/5000 images
Checked 1500/5000 images
Checked 2000/5000 images
Checked 2500/5000 images
Checked 3000/5000 images
Checked 3500/5000 images
Checked 4000/5000 images
Checked 4500/5000 images
Checked 5000/5000 images
Failure count: 0
All images decoded successfully and dimensions match the JSON.


In [ ]:
from pathlib import Path

repo = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code"
)
requirements = repo / "requirements.txt"

if not requirements.is_file():
    raise FileNotFoundError("requirements.txt was not found.")

print("--- requirements.txt ---")
print(requirements.read_text(encoding="utf-8"))

--- requirements.txt ---
# Project dependencies
# Research baseline: Python 3.13.16, Ultralytics 8.4.189.
# Recorded research PyTorch: 2.11.0+cu130 on NVIDIA Tesla T4.
# Install a platform-compatible PyTorch build separately.
# Versions below are not a fully validated environment lockfile.

ultralytics==8.4.189
torch
torchvision
numpy
matplotlib
PyYAML
pytest
pycocotools
opencv-python
Pillow



In [ ]:
import subprocess
from pathlib import Path

repo = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code"
)

result = subprocess.run(
    [
        "git", "ls-files", "--",
        "fault_injection", "fault_tolerance", "evaluation",
        "experiments", "tests", "src", "results", "docs", "models"
    ],
    cwd=repo,
    capture_output=True,
    text=True,
    check=True,
)

print(result.stdout.strip() or "No tracked files found in the selected paths.")

docs/research_reports/COCOeval_Artifact_Integrity_Manifest_v1.json
docs/research_reports/EXT2-005_Case_Study_v1.md
docs/research_reports/INT4_INT8_Bitflip_Unit_Test_Report_v1.md
docs/research_reports/Notebook_04_Closure_File_Index_v1.csv
docs/research_reports/Notebook_04_Closure_Report_v1.md
docs/research_reports/Notebook_04_Integrity_Audit_v1.json
docs/research_reports/Reproducibility_Guide_v1.md
experiments/baseline/baseline_benchmark.json
experiments/baseline/baseline_metadata.json
results/baseline_validation/baseline_coco2017_metrics.json
results/fault_injection/int4_9_cv2_conv_fp32_vs_int4_baseline_v1.json
results/fault_injection/int4_INT4-012_oracle_recovery_v1.json
results/fault_injection/int4_TMR-RAND-001_three_way_comparison_v1.json
results/fault_injection/int4_TMR-RAND-002_unprotected_v1.json
results/fault_injection/int4_TMR-RAND-003_unprotected_v1.json
results/fault_injection/int4_TMR-RAND-004_unprotected_v1.json
results/fault_injection/int4_TMR-RAND-005_unprotected_v1.json


In [ ]:
from pathlib import Path

repo = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code"
)

paths = [
    "src/int4_fault_injection.py",
    "src/int8_fault_injection.py",
    "docs/research_reports/Reproducibility_Guide_v1.md",
    "docs/research_reports/Notebook_04_Closure_Report_v1.md",
]

for relative_path in paths:
    path = repo / relative_path
    print(f"\n--- {relative_path} ---")
    if not path.is_file():
        print("NOT FOUND")
        continue
    print(path.read_text(encoding="utf-8"))


--- src/int4_fault_injection.py ---

import torch


def flip_int4_bit(value, bit_position):
    """Flip one bit in a signed 4-bit two's-complement value."""
    if not isinstance(value, int) or not -8 <= value <= 7:
        raise ValueError("INT4 value must be an integer in [-8, 7]")
    if not isinstance(bit_position, int) or not 0 <= bit_position <= 3:
        raise ValueError("bit_position must be an integer in [0, 3]")

    before_bits = value & 0xF
    after_bits = before_bits ^ (1 << bit_position)
    after = after_bits if after_bits < 8 else after_bits - 16

    record = {
        "value_before": value,
        "value_after": after,
        "bit_position": bit_position,
        "bits_before": format(before_bits, "04b"),
        "bits_after": format(after_bits, "04b"),
    }
    return after, record


def flip_int4_tensor_bit(q_tensor, index, bit_position):
    """Flip one bit in a copy of a logical signed INT4 tensor."""
    if q_tensor.dtype != torch.int8:
        raise TypeEr

In [ ]:
from pathlib import Path
import hashlib

drive_root = Path("/content/drive/MyDrive")
if not drive_root.is_dir():
    from google.colab import drive
    drive.mount("/content/drive")

checkpoint = (
    drive_root / "YOLOv8_Fault_Tolerance/models/yolov8s.pt"
)
expected_size = 22588772
expected_sha256 = (
    "1f47a78bf100391c2a140b7ac73a1caae18c32779be7d310658112f7ac9aa78a"
)

if not checkpoint.is_file():
    raise FileNotFoundError(f"Checkpoint not found: {checkpoint}")

digest = hashlib.sha256()
with checkpoint.open("rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        digest.update(chunk)

actual_size = checkpoint.stat().st_size
actual_sha256 = digest.hexdigest()

print("Checkpoint:", checkpoint)
print("Size (bytes):", actual_size)
print("SHA256:", actual_sha256)
print("Size matches recorded checkpoint:", actual_size == expected_size)
print("SHA256 matches recorded checkpoint:", actual_sha256 == expected_sha256)

if actual_size != expected_size or actual_sha256 != expected_sha256:
    raise RuntimeError("Checkpoint differs from the recorded research checkpoint.")

print("Checkpoint identity check: PASS")

Mounted at /content/drive
Checkpoint: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/models/yolov8s.pt
Size (bytes): 22588772
SHA256: 1f47a78bf100391c2a140b7ac73a1caae18c32779be7d310658112f7ac9aa78a
Size matches recorded checkpoint: True
SHA256 matches recorded checkpoint: True
Checkpoint identity check: PASS


In [ ]:
import json
from pathlib import Path
import torch

plan_path = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/"
    "results/fault_injection/int4_extended_layer_sensitivity_plan_v2.json"
)
plan = json.loads(plan_path.read_text(encoding="utf-8"))

mismatches = []

for scenario in plan["scenarios"]:
    q_clean = int(scenario["q_clean"])
    scale = float(scenario["scale"])

    baseline_value = (
        torch.tensor(q_clean, dtype=torch.float32) * scale
    ).item()

    restored_value = torch.tensor(
        float(q_clean) * scale, dtype=torch.float32
    ).item()

    if baseline_value != restored_value:
        mismatches.append({
            "scenario": scenario["scenario_id"],
            "baseline_value": baseline_value,
            "restored_value": restored_value,
            "absolute_difference": abs(baseline_value - restored_value),
        })

print("Scenarios checked:", len(plan["scenarios"]))
print("Restoration arithmetic mismatches:", len(mismatches))
for item in mismatches:
    print(item)

print("This checks arithmetic only; historical inference was not rerun.")

Scenarios checked: 96
Restoration arithmetic mismatches: 0
This checks arithmetic only; historical inference was not rerun.
